# ADMET count features and interaction Hamiltonians

Construct a reproducible family of molecular Ising Hamiltonians from hashed
Morgan **counts**, discrete feature–label mutual information, KSG pair and
conditional mutual information, and signed three-feature co-information.
The final handoff binds a selected molecule's scaled counts to local fields.

The default is an offline execution fixture: 24 public SMILES repeated four
times with invented binary labels. It exercises the real chemical and
statistical stages; these labels are not ADMET measurements. Setup, versions,
checks, and optional TDC data access are described in the project README.

The operators encode information from molecular features in chosen dimensionless
energy units (ℏ = 1). They are not an electronic-structure Hamiltonian or a
calibrated physical model of an ADMET mechanism.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "admet_curation.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the ADMET project or its pipeline directory")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from admet_curation import (
    CurationConfig, FIXTURE_NAME, public_smiles_fixture, load_tdc_datasets,
    fingerprint_datasets, select_features, estimate_pairwise,
    select_conditional_pairs, discover_triads, assemble_hamiltonians,
    molecule_problem_input,
)
from classical_preprocessing import build_problem_hamiltonian

MODE = "fixture"                    # "tdc" requires explicit opt-in below
ALLOW_DATASET_DOWNLOADS = False
DATASET = FIXTURE_NAME if MODE == "fixture" else "BBB_Martins"
SAMPLE_INDEX = 13
CONFIG = CurationConfig(top_k=4 if MODE == "fixture" else 100)
CONFIG.validate()

## 1. Data validation and molecular representation

Each retained row keeps its original position and source index. Invalid or
empty SMILES are rejected with reasons; labels must be finite binary values,
and both classes must remain. Duplicate molecular identities and conflicting
labels are reported, without silently changing observation weights.

Morgan radius 2 environments are hashed into 1,024 count features; repeated
environments retain multiplicity. Hash collisions are possible, so an index
identifies a hashed environment count rather than a unique chemical fragment.
Chirality is not included in the fingerprint (RDKit generator default).

In [ ]:
if MODE == "fixture":
    datasets = public_smiles_fixture()
elif MODE == "tdc" and ALLOW_DATASET_DOWNLOADS:
    datasets = load_tdc_datasets([DATASET], PROJECT_ROOT / "artifacts" / "tdc")
else:
    raise ValueError("Choose fixture mode, or explicitly allow TDC dataset downloads")

fingerprint_data = fingerprint_datasets(datasets, CONFIG)
fingerprints = fingerprint_data[DATASET]
display(pd.DataFrame([{
    "dataset": DATASET, "input_rows": len(datasets[DATASET]),
    "retained_rows": len(fingerprints["y"]),
    "rejected_rows": len(fingerprints["rejected_rows"]),
    "duplicate_observations": fingerprints["duplicate_observations"],
    "conflicting_label_identities": fingerprints["conflicting_label_identities"],
    "max_count": int(fingerprints["X"].max()),
    "rdkit_version": fingerprints["rdkit_version"],
}]))
display(pd.DataFrame(fingerprints["rejected_rows"]))

## 2. Discrete mutual information and molecular fields

Select features by $I(X_i;Y)$ with the discrete count estimator in
`mutual_info_classif` (**nats**). Ties use the original fingerprint index.
For each selected count column, store its min/max and map it to $[0,1]$;
constant columns map to zero.

**Fit scope: `exploratory_full_table`.** Feature selection, scaling, and
interaction discovery use the supplied table. This is Hamiltonian
construction, not a trained ADMET predictor or a held-out benchmark. A future
predictive study must freeze all fitting stages on training molecules,
prevent molecular-identity overlap, define out-of-range handling, and specify
a readout and classical baselines before estimating performance.

In [ ]:
selected_feature_data = select_features(fingerprint_data, CONFIG)
selected = selected_feature_data[DATASET]
display(pd.DataFrame({
    "selected_position": np.arange(len(selected["selected_features"])),
    "original_count_index": selected["selected_features"],
    "label_MI_nats": selected["selected_mi_scores"],
    "count_min": selected["scaling"]["feature_min"],
    "count_max": selected["scaling"]["feature_max"],
}))

## 3. Pair dependence and conditional screening

Estimate $I(X_i;X_j)$ and $I(X_i;X_j\mid Y)$ with the retained KSG estimator,
$k=3$, logarithm base 2 (**bits**). A support is retained when its clipped
conditional estimate is strictly greater than 0.10 bits. This is a heuristic
screening rule, not a significance test or a multiple-testing correction.

KSG estimates continuous densities. Applying its tiny tie-breaking jitter to
discrete, often sparse counts and binary labels is a methodological
approximation. Constants return zero; negative finite-sample pair estimates
are preserved in raw diagnostics and clipped only for nonnegative screening.
Every estimator has a reproducible seed derived from dataset, stage, and
feature support, with the caller's NumPy random state restored afterward.

In [ ]:
pairwise_mi_data = estimate_pairwise(selected_feature_data, CONFIG)
conditional_pair_data = select_conditional_pairs(selected_feature_data, CONFIG)
display(pd.DataFrame(pairwise_mi_data[DATASET]["mi_matrix"]))
display(pd.DataFrame(conditional_pair_data[DATASET]["retained_pairs"]))

## 4. Signed three-feature co-information

Extend retained pairs by another active feature and deduplicate unordered
triads. For every candidate, average the three orientations of

$$I(X_i;X_j;X_k)=I(X_i;X_j)-I(X_i;X_j\mid X_k).$$

Raw pair MI is cached consistently across orientations. Positive values
indicate redundant dependence; negative values indicate synergy. The intended
construction retains **positive** co-information above 0.15 bits and keeps
negative scores in diagnostics; it does not claim to select synergistic
interactions. The test suite checks redundant and XOR references of +1 and
−1 bit, plus an analytic correlated-Gaussian MI reference.

Pair, triad, sample, and estimator-call budgets fail before the corresponding
expensive stage. A resource failure does not truncate learned terms or replace
the estimator. Increasing budgets for a full table is an explicit experiment.

In [ ]:
triadic_interaction_data = discover_triads(conditional_pair_data, pairwise_mi_data, CONFIG)
triads = triadic_interaction_data[DATASET]
display(pd.DataFrame(triads["triad_diagnostics"]))
print({"candidate_triads": triads["num_candidate_triads"],
       "retained_triads": triads["num_retained_triads"],
       "estimator_calls_upper_bound": triads["estimator_calls_upper_bound"]})

## 5. Compact qubits and the interaction Hamiltonian

Map active selected-feature positions to contiguous qubits and retain the
original Morgan count index for each qubit. Pair CMI weights $J_{ij}$ and
positive co-information weights $K_{ijk}$ define the fixed dataset landscape

$$H_A=\sum_{i<j}J_{ij}Z_iZ_j+\sum_{i<j<k}K_{ijk}Z_iZ_jZ_k.$$

These empirical information weights define this model's dimensionless energy
scale; they are not electronic-structure energies or measured ADMET effects.

In [ ]:
hamiltonians = assemble_hamiltonians(conditional_pair_data, triadic_interaction_data)
interaction_hypergraph = hamiltonians["interaction_hypergraph"]
global_hamiltonian = hamiltonians["global_hamiltonian"]
parameterized_hamiltonian = hamiltonians["parameterized_hamiltonian"]
problem_hamiltonian = hamiltonians["problem_hamiltonian"]
graph = interaction_hypergraph[DATASET]
display(pd.DataFrame([
    {"qubit": q, "selected_position": position,
     "original_count_index": graph["original_feature_map"][q]}
    for q, position in graph["inverse_index_map"].items()
]))
display(pd.DataFrame(global_hamiltonian[DATASET]["terms"]))

## 6. Bind a molecule and construct the complete problem

For the selected source molecule, bind its scaled active counts $\theta_i$
to the one-body family and retain **all** screened interaction terms:

$$H_B(\theta)=\sum_i\theta_i Z_i,\qquad
H_{\mathrm{problem}}(\theta)=H_A+H_B(\theta).$$

Feature–label MI scores select features; they never substitute for molecular
fields. The CD notebook and the end-to-end workflow use this same adapter and
record the dataset, source row, and compact-to-original mapping.

In [ ]:
curation = {
    "schema_version": 2, "datasets": datasets, "fit_scope": "exploratory_full_table",
    "fingerprint_data": fingerprint_data, "selected_feature_data": selected_feature_data,
    "pairwise_mi_data": pairwise_mi_data, "conditional_pair_data": conditional_pair_data,
    "triadic_interaction_data": triadic_interaction_data, **hamiltonians,
}
curated_adapter, feature_vector, provenance = molecule_problem_input(curation, DATASET, SAMPLE_INDEX)
molecular_problem = build_problem_hamiltonian(curated_adapter, DATASET, feature_vector, None)
display(pd.DataFrame({"qubit": np.arange(len(feature_vector)),
                      "original_count_index": provenance["original_fingerprint_indices"],
                      "theta": feature_vector}))
display(pd.DataFrame(molecular_problem["pauli_terms"]))
display(provenance)

## 7. Exact classical spin reference and optional handoff

For this small fixture, enumerate all classical spin assignments. The diagonal
problem Hamiltonian has exactly these energies, providing a reference for
subsequent operator construction. This checks energy construction and does not
measure ADMET predictive accuracy. Large full-data models skip enumeration
above eight qubits.

The optional JSON handoff contains the bound input and provenance, without
dataset pickles, kernel state, credentials, or API calls.

In [ ]:
from itertools import product
if len(feature_vector) <= 8:
    spins = np.asarray(list(product([1.0, -1.0], repeat=len(feature_vector))))
    energies = spins @ np.asarray(feature_vector)
    for pair in graph["pair_terms"]:
        energies += pair["weight"] * spins[:, pair["qubit_i"]] * spins[:, pair["qubit_j"]]
    for triad in graph["triad_terms"]:
        energies += (triad["weight"] * spins[:, triad["qubit_i"]]
                     * spins[:, triad["qubit_j"]] * spins[:, triad["qubit_k"]])
    from quantum_reference import hamiltonian_matrix
    exact_problem = hamiltonian_matrix(molecular_problem['pauli_terms'], len(feature_vector))
    np.testing.assert_allclose(exact_problem, np.diag(np.diag(exact_problem)), atol=1e-12)
    np.testing.assert_allclose(energies, np.diag(exact_problem).real, rtol=1e-12, atol=1e-12)
    print({'classical_ground_energy': float(energies.min()), 'spin_configurations_checked': len(spins)})
    display(pd.DataFrame({"Z_eigenvalues": list(map(tuple, spins)), "energy": energies}))
else:
    print("Exact classical enumeration skipped: exceeds the eight-qubit reference budget")

WRITE_ARTIFACTS = False
if WRITE_ARTIFACTS:
    from dataclasses import asdict
    artifact = {"schema_version": 2, "mode": MODE, "fixture_labels_are_synthetic": MODE == "fixture",
                "config": asdict(CONFIG), "curated_input": curated_adapter,
                "feature_vector": feature_vector, "provenance": provenance}
    destination = PROJECT_ROOT / "artifacts" / "curated_molecule_input.json"
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(artifact, indent=2, allow_nan=False) + "\n")
    print(destination)

## 8. Measured BBB data and estimator sensitivity

The pinned public BBB artifact is a separate measured-data experiment. It has
2,039 valid observations and 1,975 canonical identities, including 64 duplicate
observations and 10 identities with conflicting labels. All rows remain in the
declared exploratory fit; no independence or held-out prediction is assumed.

Fixed top-4/top-8 profiles retain the same KSG method and thresholds. Compare
five jitter seeds with exact discrete-count CMI/co-information and descriptive
fixed-feature label shuffles. This measures estimator and graph sensitivity,
not sampling uncertainty or a calibrated significance test. The top-8 pair graph
is sensitive to jitter (pair Jaccard 0.643–0.857); the experiment reports that
limitation. Hashed count buckets may combine atom environments and are not unique
chemical substructures.

The following cell runs when `ADMET_BBB_FILE` points to the checksum-pinned raw
table, as configured in CI. With no input it performs no download. The complete
standalone experiment is `python run_benchmarks.py --download-bbb`.


In [ ]:
import os
BBB_FILE = os.environ.get('ADMET_BBB_FILE')
if BBB_FILE:
    from data_benchmark import run_data_benchmark
    benchmark_root = Path(os.environ.get('ADMET_OUTPUT_DIR', PROJECT_ROOT/'artifacts'/'runtime')).parent
    measured = run_data_benchmark(Path(BBB_FILE), output_dir=benchmark_root/'bbb_notebook')
    display({key: measured[key] for key in ['raw_rows', 'valid_rows',
        'unique_canonical_molecules', 'duplicate_observations', 'conflicting_label_identities']})
    display(pd.DataFrame([{'top_k': p['top_k'],
        'active_qubits': p['pipeline'].get('system_qubits'),
        'CD_terms': p['pipeline'].get('cd_terms'),
        'pair_threshold_disagreements': p['pair_threshold_disagreements'],
        'pair_decisions': p['pair_threshold_decisions'],
        'pipeline_status': p['pipeline']['status']} for p in measured['profiles']]))
else:
    print('No real-data file requested; reproduce the pinned BBB benchmark with run_benchmarks.py.')
